# CRMLS Data Preprocessing

## Objective

Prepare CRMLS single-family residential sales data for machine learning while
preventing data leakage and restricting model features to information that would
be available for any California property at prediction time.

The preprocessing workflow will:

1. Load and combine the monthly CRMLS datasets.
2. Restrict observations to residential single-family homes.
3. Investigate and handle duplicate and invalid records.
4. Define a leakage-safe modeling feature set.
5. Use March 2026 as the held-out test month.
6. Handle missing values and categorical encoding using training data only.
7. Export the cleaned modeling data for subsequent model development.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## 1. Load and Combine CRMLS Data

Load each monthly CRMLS sold-property dataset and preserve its source month.
Raw source files are never modified.

In [2]:
data_path = Path("../data/raw")

csv_files = sorted(data_path.glob("CRMLSSold*.csv"))

print("Files found:", len(csv_files))
print("First file:", csv_files[0].name)
print("Last file:", csv_files[-1].name)

Files found: 27
First file: CRMLSSold202401_filled.csv
Last file: CRMLSSold202603.csv


In [3]:
dataframes = []

for file in csv_files:
    monthly_df = pd.read_csv(file, low_memory=False)

    month = file.stem.replace("CRMLSSold", "").replace("_filled", "")
    monthly_df["DataMonth"] = month

    dataframes.append(monthly_df)

df = pd.concat(dataframes, ignore_index=True)

print("Combined rows:", len(df))
print("Combined columns:", df.shape[1])
print("Months:", df["DataMonth"].nunique())

Combined rows: 591446
Combined columns: 85
Months: 27


## 2. Define Modeling Population

Per the project specification, modeling is restricted to:

- `PropertyType == "Residential"`
- `PropertySubType == "SingleFamilyResidence"`

In [4]:
df_sfr = df[
    (df["PropertyType"] == "Residential")
    & (df["PropertySubType"] == "SingleFamilyResidence")
].copy()

print("Rows before SFR filter:", len(df))
print("Rows after SFR filter:", len(df_sfr))
print("Percent retained:", round(len(df_sfr) / len(df) * 100, 2), "%")

Rows before SFR filter: 591446
Rows after SFR filter: 297245
Percent retained: 50.26 %


## 3. Duplicate and Data-Quality Investigation

Before removing duplicate records, inspect repeated observations to determine
whether they represent duplicate data or legitimate separate transactions.

Exact duplicate rows can generally be removed safely. Repeated `ListingKey`
values require additional investigation because the same listing may appear
across multiple monthly source files.

In [5]:
print("Exact duplicate rows:", df_sfr.duplicated().sum())

print(
    "Duplicated ListingKeys:",
    df_sfr["ListingKey"].duplicated().sum()
)

print(
    "Rows involved in repeated ListingKeys:",
    df_sfr["ListingKey"].duplicated(keep=False).sum()
)

Exact duplicate rows: 29
Duplicated ListingKeys: 258
Rows involved in repeated ListingKeys: 512


In [6]:
duplicate_listing_rows = (
    df_sfr[
        df_sfr["ListingKey"].duplicated(keep=False)
    ]
    .sort_values(["ListingKey", "DataMonth"])
)

duplicate_listing_rows[
    [
        "ListingKey",
        "DataMonth",
        "CloseDate",
        "ClosePrice",
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "City",
        "PostalCode"
    ]
].head(30)

,ListingKey,DataMonth,CloseDate,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,City,PostalCode
136532,1031992525,202406,2024-06-21,415740.0,1760.0,4.0,2.0,Brawley,92227
231924,1031992525,202410,2024-10-18,459900.0,1760.0,4.0,2.0,Brawley,92227
112046,1037464932,202405,2024-05-25,650000.0,1012.0,2.0,2.0,Whittier,90601
208611,1037464932,202409,2024-09-06,665000.0,1012.0,2.0,2.0,Whittier,90601
135852,1046689006,202406,2024-06-30,825000.0,1248.0,3.0,2.0,Grover Beach,93433
231725,1046689006,202410,2024-10-29,825000.0,1248.0,3.0,2.0,Grover Beach,93433
135638,1048638478,202406,2024-06-25,6600000.0,5621.0,4.0,5.0,Hidden Hills,91302
186923,1048638478,202408,2024-08-21,6550000.0,5621.0,4.0,5.0,Hidden Hills,91302
82994,1052182218,202404,2024-04-26,355000.0,1464.0,2.0,2.0,Lake Hughes,93532
186886,1052182218,202408,2024-08-20,355000.0,1464.0,2.0,2.0,Lake Hughes,93532


### Duplicate Handling Decision

Repeated `ListingKey` values do not consistently represent identical records.
Several repeated listing keys have different close dates or close prices, suggesting
that they may represent separate transactions, relistings, or updates in the MLS data.

Therefore, records will not be removed solely because they share a `ListingKey`.

Exact duplicate rows, however, contain no additional information and will be removed.

In [7]:
rows_before = len(df_sfr)

df_sfr = df_sfr.drop_duplicates().copy()

rows_after = len(df_sfr)

print("Rows before exact deduplication:", rows_before)
print("Rows after exact deduplication:", rows_after)
print("Rows removed:", rows_before - rows_after)

Rows before exact deduplication: 297245
Rows after exact deduplication: 297216
Rows removed: 29


In [8]:
transaction_key = [
    "ListingKey",
    "CloseDate",
    "ClosePrice"
]

print(
    "Duplicate ListingKey + CloseDate + ClosePrice:",
    df_sfr.duplicated(subset=transaction_key).sum()
)

Duplicate ListingKey + CloseDate + ClosePrice: 0


In [9]:
transaction_duplicates = df_sfr[
    df_sfr.duplicated(
        subset=transaction_key,
        keep=False
    )
].sort_values(transaction_key)

transaction_duplicates[
    [
        "ListingKey",
        "DataMonth",
        "CloseDate",
        "ClosePrice",
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "City",
        "PostalCode"
    ]
].head(30)

,ListingKey,DataMonth,CloseDate,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,City,PostalCode


**Decision:** After removing exact duplicate rows, no duplicate combinations of
`ListingKey`, `CloseDate`, and `ClosePrice` remained. Repeated `ListingKey`
values were therefore retained because they correspond to records with different
transaction information.

## 4. Invalid and Implausible Values

The EDA identified zero values and extreme observations in several important
modeling variables. Before handling missing values, these records are examined
to distinguish invalid data from legitimate but unusual properties.

In [10]:
core_numeric = [
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet"
]

invalid_summary = pd.DataFrame({
    "Missing": df_sfr[core_numeric].isna().sum(),
    "Zero": (df_sfr[core_numeric] == 0).sum(),
    "Negative": (df_sfr[core_numeric] < 0).sum(),
    "Min": df_sfr[core_numeric].min(),
    "Max": df_sfr[core_numeric].max()
})

invalid_summary

,Missing,Zero,Negative,Min,Max
ClosePrice,2,1,0,0.0,9.895000e+08
LivingArea,161,121,0,0.0,1.237640e+05
BedroomsTotal,0,173,0,0.0,4.500000e+01
BathroomsTotalInteger,50,128,0,0.0,1.750000e+02
LotSizeSquareFeet,5082,151,0,0.0,2.087221e+09


### Distribution of Extreme Values

Extreme observations are examined before defining cleaning rules. Legitimate
high-value or unusually large California properties should not be removed solely
because they are statistical outliers.

In [12]:
quantiles = [0, 0.001, 0.005, 0.01, 0.50, 0.99, 0.995, 0.999, 1.0]

df_sfr[core_numeric].quantile(quantiles).T

,0.000,0.001,0.005,0.010,0.500,0.990,0.995,0.999,1.000
ClosePrice,0.0,91213.00,190000.0,238000.0,890000.0,6239608.713,8400000.00,1.700000e+07,9.895000e+08
LivingArea,0.0,440.00,636.0,728.0,1802.0,5648.000,6753.46,1.006314e+04,1.237640e+05
BedroomsTotal,0.0,1.00,1.0,2.0,3.0,6.000,6.00,8.000000e+00,4.500000e+01
BathroomsTotalInteger,0.0,1.00,1.0,1.0,2.0,6.000,7.00,1.000000e+01,1.750000e+02
LotSizeSquareFeet,0.0,0.18,1368.0,1796.0,7245.0,248727.600,473120.29,3.484800e+06,2.087221e+09


In [13]:
df_sfr.nlargest(15, "ClosePrice")[
    [
        "ClosePrice",
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "City",
        "PostalCode",
        "CloseDate"
    ]
]

,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,City,PostalCode,CloseDate
475560,989500000.0,2289.0,5.0,3.0,10256.00,Valencia,91355,2025-10-31
350899,970000000.0,1976.0,4.0,3.0,5344.00,Chula Vista,91913,2025-04-11
400421,900000000.0,11000.0,13.0,10.0,13072356.00,Glennville,93287,2025-06-27
447352,875000000.0,1730.0,4.0,2.0,8900.00,San Diego,92117,2025-09-25
454029,835800000.0,718.0,1.0,1.0,549727.20,Ramona,92065,2025-09-11
464634,815000000.0,1900.0,4.0,3.0,6016.00,Saugus,91350,2025-09-10
448736,751412000.0,798.0,2.0,1.0,4791.60,La Mesa,91942,2025-09-26
507181,740000000.0,1271.0,3.0,2.0,5100.00,San Diego,92154,2025-11-24
585671,664250000.0,1992.0,4.0,3.0,0.23,Chula Vista,91911,2026-03-26
407707,613000000.0,728.0,2.0,1.0,6417.00,La Mesa,91941,2025-07-28


In [14]:
price_thresholds = [
    10_000_000,
    20_000_000,
    50_000_000,
    100_000_000
]

for threshold in price_thresholds:
    count = (df_sfr["ClosePrice"] > threshold).sum()
    percent = count / len(df_sfr) * 100

    print(
        f"Above ${threshold:,.0f}: "
        f"{count:,} rows ({percent:.4f}%)"
    )

Above $10,000,000: 991 rows (0.3334%)
Above $20,000,000: 194 rows (0.0653%)
Above $50,000,000: 27 rows (0.0091%)
Above $100,000,000: 20 rows (0.0067%)


In [15]:
df_sfr[df_sfr["ClosePrice"] > 100_000_000][
    [
        "ClosePrice",
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "City",
        "PostalCode",
        "CloseDate"
    ]
].sort_values("ClosePrice", ascending=False)

,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,City,PostalCode,CloseDate
475560,989500000.0,2289.0,5.0,3.0,10256.00,Valencia,91355,2025-10-31
350899,970000000.0,1976.0,4.0,3.0,5344.00,Chula Vista,91913,2025-04-11
400421,900000000.0,11000.0,13.0,10.0,13072356.00,Glennville,93287,2025-06-27
447352,875000000.0,1730.0,4.0,2.0,8900.00,San Diego,92117,2025-09-25
454029,835800000.0,718.0,1.0,1.0,549727.20,Ramona,92065,2025-09-11
464634,815000000.0,1900.0,4.0,3.0,6016.00,Saugus,91350,2025-09-10
448736,751412000.0,798.0,2.0,1.0,4791.60,La Mesa,91942,2025-09-26
507181,740000000.0,1271.0,3.0,2.0,5100.00,San Diego,92154,2025-11-24
585671,664250000.0,1992.0,4.0,3.0,0.23,Chula Vista,91911,2026-03-26
407707,613000000.0,728.0,2.0,1.0,6417.00,La Mesa,91941,2025-07-28


### Target Cleaning

`ClosePrice` is the prediction target and therefore cannot be imputed.

Records with missing or non-positive close prices are removed. Inspection of the
upper tail also identified a small number of implausible values above $100 million,
including several extremely high prices associated with otherwise ordinary-sized
single-family homes.

A conservative upper bound of $100 million is therefore used as a data-quality
filter. This preserves legitimate luxury transactions while removing only the most
extreme and likely erroneous observations.

In [16]:
rows_before = len(df_sfr)

df_sfr = df_sfr[
    df_sfr["ClosePrice"].notna()
    & (df_sfr["ClosePrice"] > 0)
    & (df_sfr["ClosePrice"] <= 100_000_000)
].copy()

print("Rows before target cleaning:", rows_before)
print("Rows after target cleaning:", len(df_sfr))
print("Rows removed:", rows_before - len(df_sfr))

Rows before target cleaning: 297216
Rows after target cleaning: 297193
Rows removed: 23


### Invalid Predictor Values

Unlike the target variable, observations are not removed solely because a predictor
contains an invalid or missing value. Zero values in key physical-property fields
are treated as missing where they are not meaningful measurements.

These missing values will later be imputed using statistics learned from the
training data only, preventing information from the test period from leaking into
model training.

In [17]:
zero_as_missing = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet"
]

for column in zero_as_missing:
    df_sfr.loc[df_sfr[column] == 0, column] = np.nan

df_sfr[zero_as_missing].isna().sum()

LivingArea                281
BedroomsTotal             173
BathroomsTotalInteger     178
LotSizeSquareFeet        5233
dtype: int64

## 5. Temporal Train/Test Split

Because the model is intended to predict future property values, a chronological
split is used instead of a random train/test split.

Per the project specification, the most recent available month (March 2026) is
reserved as the test set. Earlier months are available for model training.

The length of the training window will later be evaluated as a modeling choice.
All preprocessing parameters that depend on the data will be learned from the
training set only.

In [18]:
df_sfr["DataMonth"] = df_sfr["DataMonth"].astype(str)

test_month = "202603"

train_df = df_sfr[df_sfr["DataMonth"] < test_month].copy()
test_df = df_sfr[df_sfr["DataMonth"] == test_month].copy()

print("Training period:", train_df["DataMonth"].min(), "to", train_df["DataMonth"].max())
print("Test period:", test_df["DataMonth"].min(), "to", test_df["DataMonth"].max())

print()
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print()
print("Test month overlap in training:", (train_df["DataMonth"] == test_month).sum())

Training period: 202401 to 202602
Test period: 202603 to 202603

Training rows: 285632
Test rows: 11561

Test month overlap in training: 0


## 6. Modeling Feature Set

Model inputs are restricted to property characteristics that would be available
for any California single-family property at prediction time, including properties
that are not currently listed for sale.

Listing-dependent variables such as `ListPrice`, `OriginalListPrice`,
`DaysOnMarket`, `PurchaseContractDate`, and `ListingContractDate` are excluded
from the modeling feature set to prevent target leakage and ensure the model can
be applied to off-market properties.

The initial preprocessing pipeline uses core physical and geographic property
characteristics. Additional features will be evaluated during the feature
engineering stage.

In [19]:
numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet"
]

categorical_features = [
    "City",
    "PostalCode"
]

target = "ClosePrice"

model_features = numeric_features + categorical_features

X_train = train_df[model_features].copy()
X_test = test_df[model_features].copy()

y_train = train_df[target].copy()
y_test = test_df[target].copy()

print("Model features:", model_features)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Model features: ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'City', 'PostalCode']
X_train shape: (285632, 6)
X_test shape: (11561, 6)


In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 7. Training-Only Preprocessing

Preprocessing transformations are fitted using the training data only.

Numeric features are median-imputed and standardized. Categorical features are
imputed with the most frequent training value and one-hot encoded. Unknown
categories encountered in the March 2026 test set are ignored safely.

Using a scikit-learn preprocessing pipeline ensures that statistics from the
held-out test month do not influence model preparation.

In [23]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (285632, 4142)
Processed test shape: (11561, 4142)


## 8. Export Cleaned Modeling Data

The cleaned chronological train and test datasets are exported for reproducible
model development in subsequent notebooks.

In [24]:
processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

train_output = train_df[model_features + [target, "DataMonth"]]
test_output = test_df[model_features + [target, "DataMonth"]]

train_output.to_csv(
    processed_path / "crmls_train_cleaned.csv",
    index=False
)

test_output.to_csv(
    processed_path / "crmls_test_202603_cleaned.csv",
    index=False
)

print("Saved:", processed_path / "crmls_train_cleaned.csv")
print("Saved:", processed_path / "crmls_test_202603_cleaned.csv")

Saved: ../data/processed/crmls_train_cleaned.csv
Saved: ../data/processed/crmls_test_202603_cleaned.csv


## Preprocessing Summary

The Week 3 preprocessing workflow:

- Combined 27 monthly CRMLS datasets covering January 2024 through March 2026.
- Restricted the modeling population to residential single-family homes.
- Removed 29 exact duplicate records after investigating repeated `ListingKey` values.
- Retained repeated listing keys when transaction information differed.
- Removed records with missing or non-positive `ClosePrice`.
- Removed a small number of implausible `ClosePrice` observations above $100 million.
- Converted invalid zero values in key physical-property fields to missing values.
- Excluded listing-dependent features that would not be available for an off-market property.
- Reserved March 2026 as the held-out test month.
- Prepared numeric and categorical preprocessing using training data only to prevent leakage.
- Exported cleaned chronological train and test datasets for baseline modeling.

The next stage will train and evaluate a Linear Regression baseline model using
the preprocessing pipeline developed here.